<a href="https://colab.research.google.com/github/qhamatul/PROJEK_AICV_2026/blob/main/KALKULATOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BUAT KALKULATOR

In [ ]:
import ast
import math
import operator
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- LOGIK PENGIRAAN MATEMATIK (PYTHON TULEN) ---
expression = ""

ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos
}

ALLOWED_FUNCTIONS = {
    'sin': lambda x: math.sin(math.radians(x)),
    'cos': lambda x: math.cos(math.radians(x)),
    'tan': lambda x: math.tan(math.radians(x)),
    'log': math.log10,
    'ln': math.log,
    'sqrt': math.sqrt,
    'pi': math.pi,
    'e': math.e
}

def safe_eval(expr_str):
    parsed_str = expr_str.replace('×', '*').replace('÷', '/')
    parsed_str = parsed_str.replace('^', '**').replace('√', 'sqrt')

    # Auto-close kurung
    open_b = parsed_str.count('(')
    close_b = parsed_str.count(')')
    if open_b > close_b:
        parsed_str += ')' * (open_b - close_b)

    tree = ast.parse(parsed_str, mode='eval')
    return _eval(tree.body)

def _eval(node):
    if isinstance(node, ast.Num):
        return node.n
    elif isinstance(node, ast.BinOp):
        return ALLOWED_OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
    elif isinstance(node, ast.UnaryOp):
        return ALLOWED_OPERATORS[type(node.op)](_eval(node.operand))
    elif isinstance(node, ast.Call):
        func_name = node.func.id
        if func_name in ALLOWED_FUNCTIONS:
            args = [_eval(arg) for arg in node.args]
            return ALLOWED_FUNCTIONS[func_name](*args)
        raise NameError(f"Fungsi '{func_name}' tidak didukung.")
    elif isinstance(node, ast.Name):
        if node.id in ['pi', 'e']:
            return ALLOWED_FUNCTIONS[node.id]
        raise NameError(f"Variabel '{node.id}' tidak dikenal.")
    else:
        raise TypeError(node)

# --- ANTARAMUKA GRAFIK (IPYWIDGETS - TIADA HTML) ---

# 1. Cipta Skrin Paparan (Kotak Teks)
display_screen = widgets.Text(
    value='0',
    disabled=True,
    layout=widgets.Layout(width='290px', margin='10px 0px 10px 0px')
)

# 2. Fungsi Kesan Tekanan Butang
def on_button_click(b):
    global expression
    val = b.description

    if val == 'AC':
        expression = ""
        display_screen.value = "0"
    elif val == '⌫':
        for f in ['sin(', 'cos(', 'tan(', 'log(', 'ln(', '√(']:
            if expression.endswith(f):
                expression = expression[:-len(f)]
                display_screen.value = expression if expression else "0"
                return
        expression = expression[:-1]
        display_screen.value = expression if expression else "0"
    elif val == '=':
        if not expression:
            display_screen.value = "0"
            return
        try:
            result = safe_eval(expression)
            if isinstance(result, float):
                expression = str(int(result) if result.is_integer() else round(result, 6))
            else:
                expression = str(result)
            display_screen.value = expression
        except ZeroDivisionError:
            expression = ""
            display_screen.value = "Error: Div by 0"
        except Exception:
            expression = ""
            display_screen.value = "Syntax Error"
    elif val in ['sin', 'cos', 'tan', 'log', 'ln', '√']:
        expression += val + "("
        display_screen.value = expression
    elif val == 'xʸ':
        expression += '^'
        display_screen.value = expression
    else:
        expression += val
        display_screen.value = expression

# 3. Senarai Susunan Butang dan Gaya Warna Baru (Gaya Cyber/Moden Melalui Pilihan Warna Widget)
buttons_layout = [
    ['sin', 'cos', 'tan', 'π'],
    ['log', 'ln', '√', 'xʸ'],
    ['(', ')', 'e', '⌫'],
    ['7', '8', '9', '÷'],
    ['4', '5', '6', '×'],
    ['1', '2', '3', '-'],
    ['0', '.', 'AC', '+'],
    ['=']
]

grid_rows = []

for row in buttons_layout:
    row_widgets = []
    for label in row:
        # Tentukan saiz butang (Butang '=' akan dipanjangkan)
        btn_width = '290px' if label == '=' else '65px'

        # Tentukan gaya warna asas menggunakan sistem "button_style" bawaan widget
        b_style = ''
        if label in ['sin', 'cos', 'tan', 'log', 'ln', '√', 'π', 'e', 'xʸ']:
            b_style = 'info'     # Biru Cyan
        elif label in ['÷', '×', '-', '+', '(', ')']:
            b_style = 'primary'  # Biru Gelap
        elif label in ['AC', '⌫']:
            b_style = 'danger'   # Merah / Oren
        elif label == '=':
            b_style = 'success'  # Hijau

        btn = widgets.Button(
            description=label,
            layout=widgets.Layout(width=btn_width, height='45px', margin='2px'),
            button_style=b_style
        )
        btn.on_click(on_button_click)
        row_widgets.append(btn)

    grid_rows.append(widgets.HBox(row_widgets))

# 4. Satukan Skrin dan Butang ke Dalam Kotak Utama (VBox)
calculator_vbox = widgets.VBox([display_screen] + grid_rows, layout=widgets.Layout(
    align_items='center',
    border='2px solid #3b0764',
    padding='10px',
    width='320px',
    background_color='#0f0c1b'
))

# Paparkan kalkulator di Google Colab
clear_output()
display(calculator_vbox)

/tmp/ipykernel_490/1377584862.py:45: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):
/tmp/ipykernel_490/1377584862.py:46: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  return node.n
